# Exercice 6 : Stockage dans HDFS organisé

## connexion à HDFS et KAFKA

In [1]:
from kafka import KafkaConsumer
from hdfs import InsecureClient
import json

TOPIC = "weather_transformed"
DOSSIER_HDFS = "/hdfs-data"

client = InsecureClient("http://namenode:9870", user="root")

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers="kafka:9092",
    group_id="archivage_hdfs",
    auto_offset_reset="earliest",
    value_deserializer=lambda v: v.decode("utf-8")
)

print("Connecté à HDFS, contenu de la racine :", client.list("/"))


Connecté à HDFS, contenu de la racine : ['rmstate']


## ajout des lignes dans HDFS

In [2]:
def ajouter_dans_hdfs(ligne_json, country, city):
    dossier = DOSSIER_HDFS + "/" + country + "/" + city
    fichier = dossier + "/alerts.json"

    if client.status(fichier, strict=False) is None:
        # le fichier n'existe pas encore : on le crée
        client.makedirs(dossier)
        client.write(fichier, data=ligne_json + "\n", encoding="utf-8", replication=1)
    else:
        # le fichier existe : on ajoute la ligne à la fin
        client.write(fichier, data=ligne_json + "\n", encoding="utf-8", append=True)

    return fichier


## Boucle d'archivage


In [3]:
nb_messages = 0
print("Archivage de", TOPIC, "vers HDFS", DOSSIER_HDFS, "...")

try:
    for message in consumer:
        texte = message.value
        try:
            data = json.loads(texte)
        except json.JSONDecodeError:
            print("message ignoré (pas du JSON) :", repr(texte))
            continue

        fichier = ajouter_dans_hdfs(texte, data["country"], data["city"])
        nb_messages = nb_messages + 1
        print(nb_messages, "|", data["event_time"], "|", data["city"],
              "| vent", data["wind_alert_level"], "| chaleur", data["heat_alert_level"],
              "->", fichier)
except KeyboardInterrupt:
    print("Archivage arrêté. Messages écrits dans HDFS :", nb_messages)


Archivage de weather_transformed vers HDFS /hdfs-data ...
1 | 2026-11-10T14:51:48.428Z | Paris | vent level_0 | chaleur level_0 -> /hdfs-data/France/Paris/alerts.json
2 | 2026-11-10T14:51:48.540Z | Seville | vent level_0 | chaleur level_0 -> /hdfs-data/Spain/Seville/alerts.json
3 | 2026-11-10T14:51:48.659Z | Dubai | vent level_0 | chaleur level_1 -> /hdfs-data/United Arab Emirates/Dubai/alerts.json
4 | 2026-11-10T14:51:48.780Z | Reykjavik | vent level_0 | chaleur level_0 -> /hdfs-data/Iceland/Reykjavik/alerts.json
5 | 2026-11-10T14:51:48.888Z | Wellington | vent level_0 | chaleur level_0 -> /hdfs-data/New Zealand/Wellington/alerts.json
6 | 2026-11-10T14:52:19.605Z | Paris | vent level_0 | chaleur level_0 -> /hdfs-data/France/Paris/alerts.json
7 | 2026-11-10T14:52:19.833Z | Seville | vent level_0 | chaleur level_0 -> /hdfs-data/Spain/Seville/alerts.json
8 | 2026-11-10T14:52:19.990Z | Dubai | vent level_0 | chaleur level_1 -> /hdfs-data/United Arab Emirates/Dubai/alerts.json
9 | 2026-11-

In [4]:
for dossier, sous_dossiers, fichiers in client.walk(DOSSIER_HDFS):
    for f in fichiers:
        chemin = dossier + "/" + f
        with client.read(chemin, encoding="utf-8") as lecteur:
            lignes = lecteur.read().splitlines()
        print(chemin, "->", len(lignes), "lignes")

print()
print("Les 2 dernières lignes de Paris :")
with client.read(DOSSIER_HDFS + "/France/Paris/alerts.json", encoding="utf-8") as lecteur:
    lignes = lecteur.read().splitlines()
for ligne in lignes[-2:]:
    print(ligne)


/hdfs-data/France/Paris/alerts.json -> 90 lignes
/hdfs-data/Iceland/Reykjavik/alerts.json -> 89 lignes
/hdfs-data/New Zealand/Wellington/alerts.json -> 89 lignes
/hdfs-data/Spain/Seville/alerts.json -> 90 lignes
/hdfs-data/United Arab Emirates/Dubai/alerts.json -> 89 lignes

Les 2 dernières lignes de Paris :
{"event_time":"2026-11-10T15:37:00.204Z","city":"Paris","country":"France","temperature":12.5,"windspeed":1.72,"weather_code":3,"wind_alert_level":"level_0","heat_alert_level":"level_0"}
{"event_time":"2026-11-10T15:37:31.476Z","city":"Paris","country":"France","temperature":12.5,"windspeed":1.72,"weather_code":3,"wind_alert_level":"level_0","heat_alert_level":"level_0"}
